# In Class Assignment 2026.04.23

In [17]:
import numpy as np
import pandas as pd

from feature_engine.encoding import RareLabelEncoder, CountFrequencyEncoder
from feature_engine.discretisation import EqualFrequencyDiscretiser
from feature_engine.selection import DropConstantFeatures

from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.pipeline import Pipeline

import xgboost as xgb

from scipy.stats import randint, loguniform

In [5]:
adult = pd.read_csv("adult.csv")

numeric_cols = ["age", "fnlwgt", "educational-num", "capital-gain", "capital-loss", "hours-per-week"]
adult[numeric_cols] = adult[numeric_cols].apply(pd.to_numeric, errors="coerce")

adult["income"] = adult["income"].map({">50K": 1, "<=50K": 0})
adult["gender"] = adult["gender"].map({"Male": 1, "Female": 0})
adult = adult.drop(columns=["fnlwgt"]).replace("?", np.nan)

for col in adult.columns:
    if col == "income":
        continue
    if pd.api.types.is_numeric_dtype(adult[col]):
        adult[col] = adult[col].fillna(adult[col].median())
    else:
        adult[col] = adult[col].fillna("unknown")

adult.head()

,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,11th,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States,0
1,38,Private,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States,0
2,28,Local-gov,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States,1
3,44,Private,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States,1
4,18,unknown,Some-college,10,Never-married,unknown,Own-child,White,0,0,0,30,United-States,0


## Reducing features from 377 to at most 20

In [6]:
X = adult.drop(columns=["income"])
y = adult["income"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

cat_cols = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

rare_encoder = RareLabelEncoder(tol=0.01, variables=cat_cols)
freq_encoder = CountFrequencyEncoder(variables=cat_cols, encoding_method="frequency")
disc_vars = [c for c in num_cols if c not in ["gender", "capital-gain", "capital-loss"]]
disc = EqualFrequencyDiscretiser(q=5, variables=disc_vars)
const_drop = DropConstantFeatures()

X_train_fe = rare_encoder.fit_transform(X_train)
X_test_fe = rare_encoder.transform(X_test)

X_train_fe = freq_encoder.fit_transform(X_train_fe)
X_test_fe = freq_encoder.transform(X_test_fe)

X_train_fe = disc.fit_transform(X_train_fe)
X_test_fe = disc.transform(X_test_fe)

X_train_fe = const_drop.fit_transform(X_train_fe)
X_test_fe = const_drop.transform(X_test_fe)

print("Transformed feature-engine shape:", X_train_fe.shape)
X_train_fe.head()


c:\Users\jfigg\AppData\Local\Programs\Python\Python314\Lib\site-packages\feature_engine\encoding\rare_label.py:216: UserWarning: The number of unique categories for variable workclass is less than that indicated in n_categories. Thus, all categories will be considered frequent
  warnings.warn(
c:\Users\jfigg\AppData\Local\Programs\Python\Python314\Lib\site-packages\feature_engine\encoding\rare_label.py:216: UserWarning: The number of unique categories for variable marital-status is less than that indicated in n_categories. Thus, all categories will be considered frequent
  warnings.warn(
c:\Users\jfigg\AppData\Local\Programs\Python\Python314\Lib\site-packages\feature_engine\encoding\rare_label.py:216: UserWarning: The number of unique categories for variable relationship is less than that indicated in n_categories. Thus, all categories will be considered frequent
  warnings.warn(
c:\Users\jfigg\AppData\Local\Programs\Python\Python314\Lib\site-packages\feature_engine\encoding\rare_label

Transformed feature-engine shape: (39073, 13)


,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country
34342,4,0.693727,0.324751,0,0.330945,0.042664,0.258618,0.85609,1,0,0,0,0.898344
18559,0,0.693727,0.028613,0,0.330945,0.113122,0.031352,0.85609,0,0,0,0,0.898344
12477,1,0.693727,0.324751,0,0.456632,0.101144,0.402580,0.85609,1,0,0,1,0.064699
560,3,0.693727,0.324751,0,0.031684,0.115067,0.105239,0.09513,0,0,0,1,0.898344
3427,1,0.693727,0.164257,2,0.456632,0.125278,0.402580,0.85609,1,0,0,1,0.898344


In [7]:
poly = PolynomialFeatures(degree=3, interaction_only=True, include_bias=False)

X_train_poly = pd.DataFrame(
    poly.fit_transform(X_train_fe),
    columns=poly.get_feature_names_out(X_train_fe.columns),
    index=X_train_fe.index
)

X_test_poly = pd.DataFrame(
    poly.transform(X_test_fe),
    columns=poly.get_feature_names_out(X_train_fe.columns),
    index=X_test_fe.index
)

print("Expanded train shape:", X_train_poly.shape)
print("Expanded test shape:", X_test_poly.shape)


Expanded train shape: (39073, 377)
Expanded test shape: (9769, 377)


In [8]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

rf_selector = RandomForestClassifier(
    n_estimators=150,
    class_weight="balanced",
    min_samples_leaf=2,
    random_state=42,
    n_jobs=1
)

rf_selector.fit(X_train_poly, y_train)

rf_perm = permutation_importance(
    rf_selector,
    X_test_poly,
    y_test,
    n_repeats=3,
    random_state=42,
    scoring="balanced_accuracy"
)

feature_rank = pd.DataFrame({
    "feature": X_train_poly.columns,
    "rf_importance": rf_selector.feature_importances_,
    "perm_importance": rf_perm.importances_mean
})

feature_rank["rf_rank"] = feature_rank["rf_importance"].rank(ascending=False)
feature_rank["perm_rank"] = feature_rank["perm_importance"].rank(ascending=False)
feature_rank["avg_rank"] = (feature_rank["rf_rank"] + feature_rank["perm_rank"]) / 2

feature_rank = feature_rank.sort_values("avg_rank")
feature_rank.head(20)


,feature,rf_importance,perm_importance,rf_rank,perm_rank,avg_rank
4,marital-status,0.052772,0.004087,1.0,1.0,1.0
55,marital-status occupation,0.013517,0.002206,9.0,2.0,5.5
121,age marital-status occupation,0.017662,0.000991,6.0,20.0,13.0
258,educational-num marital-status relationship,0.008287,0.001227,22.0,10.0,16.0
270,educational-num occupation hours-per-week,0.008120,0.001190,24.0,13.0,18.5
310,marital-status race native-country,0.009748,0.000932,13.0,25.0,19.0
322,occupation relationship gender,0.007838,0.001135,26.0,16.0,21.0
276,educational-num relationship hours-per-week,0.006491,0.001193,32.0,12.0,22.0
112,age educational-num marital-status,0.017810,0.000628,5.0,44.0,24.5
149,age gender hours-per-week,0.005579,0.001371,44.0,6.0,25.0


In [9]:
top_10_features = feature_rank.head(10)["feature"].tolist()
top_20_features = feature_rank.head(20)["feature"].tolist()

X_train_10 = X_train_poly[top_10_features]
X_test_10 = X_test_poly[top_10_features]

X_train_20 = X_train_poly[top_20_features]
X_test_20 = X_test_poly[top_20_features]

print("10-feature set:", len(top_10_features))
print("20-feature set:", len(top_20_features))


10-feature set: 10
20-feature set: 20


## Training 2 models that differ meaningfully

In [10]:
rf_base = RandomForestClassifier(
    n_estimators=150,
    class_weight="balanced",
    min_samples_leaf=2,
    random_state=42,
    n_jobs=1
)

lr_base = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="liblinear",
        random_state=42
    ))
])

results = []

for label, Xtr, Xte in [
    ("377 features", X_train_poly, X_test_poly),
    ("20 features", X_train_20, X_test_20),
    ("10 features", X_train_10, X_test_10),
]:
    for model_name, model in [("Random Forest", rf_base), ("Logistic Regression", lr_base)]:
        cv_score = cross_val_score(model, Xtr, y_train, cv=skf, scoring="balanced_accuracy", n_jobs=1).mean()
        model.fit(Xtr, y_train)
        test_score = balanced_accuracy_score(y_test, model.predict(Xte))
        results.append([label, model_name, cv_score, test_score])

results_df = pd.DataFrame(results, columns=["feature_set", "model", "cv_bal_acc", "test_bal_acc"])
results_df


,feature_set,model,cv_bal_acc,test_bal_acc
0,377 features,Random Forest,0.803396,0.810313
1,377 features,Logistic Regression,0.812844,0.813756
2,20 features,Random Forest,0.785880,0.783267
3,20 features,Logistic Regression,0.787621,0.788701
4,10 features,Random Forest,0.785753,0.791669
5,10 features,Logistic Regression,0.770762,0.775942


## Tuning models beyond default settings

In [11]:
rf_search = RandomizedSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=1),
    param_distributions={
        "n_estimators": randint(100, 300),
        "max_depth": [None, 6, 8, 10, 14],
        "min_samples_split": randint(2, 10),
        "min_samples_leaf": randint(1, 5),
        "max_features": ["sqrt", "log2", 0.5]
    },
    n_iter=5,
    scoring="balanced_accuracy",
    cv=skf,
    random_state=42,
    n_jobs=1
)

rf_search.fit(X_train_20, y_train)
rf_best = rf_search.best_estimator_

print(rf_search.best_score_)
print(rf_search.best_params_)


0.8065311982170756
{'max_depth': 8, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 9, 'n_estimators': 251}


In [12]:
lr_search = RandomizedSearchCV(
    Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            solver="liblinear",
            random_state=42
        ))
    ]),
    param_distributions={
        "clf__C": loguniform(0.01, 10),
        "clf__penalty": ["l1", "l2"]
    },
    n_iter=5,
    scoring="balanced_accuracy",
    cv=skf,
    random_state=42,
    n_jobs=1
)

lr_search.fit(X_train_20, y_train)
lr_best = lr_search.best_estimator_

print(lr_search.best_score_)
print(lr_search.best_params_)


0.7874649523768179
{'clf__C': np.float64(0.6251373574521749), 'clf__penalty': 'l1'}


In [13]:
tuned_results = []

for model_name, model in [("Tuned Random Forest", rf_best), ("Tuned Logistic Regression", lr_best)]:
    model.fit(X_train_20, y_train)
    preds = model.predict(X_test_20)
    score = balanced_accuracy_score(y_test, preds)
    tuned_results.append([model_name, score])
    print(model_name)
    print("Balanced accuracy:", score)
    print(classification_report(y_test, preds))

pd.DataFrame(tuned_results, columns=["model", "test_bal_acc"])


Tuned Random Forest
Balanced accuracy: 0.8066025512847654
              precision    recall  f1-score   support

           0       0.94      0.78      0.85      7431
           1       0.55      0.83      0.66      2338

    accuracy                           0.79      9769
   macro avg       0.74      0.81      0.76      9769
weighted avg       0.84      0.79      0.81      9769

Tuned Logistic Regression
Balanced accuracy: 0.7885424721236344
              precision    recall  f1-score   support

           0       0.92      0.78      0.85      7431
           1       0.54      0.79      0.64      2338

    accuracy                           0.79      9769
   macro avg       0.73      0.79      0.74      9769
weighted avg       0.83      0.79      0.80      9769



,model,test_bal_acc
0,Tuned Random Forest,0.806603
1,Tuned Logistic Regression,0.788542


## Combining models with stacking or an OOF meta-learner

In [19]:
rf_oof = cross_val_predict(
    rf_best, X_train_20, y_train, cv=skf, method="predict_proba", n_jobs=1
)[:, 1]

lr_oof = cross_val_predict(
    lr_best, X_train_20, y_train, cv=skf, method="predict_proba", n_jobs=1
)[:, 1]

X_meta_train = np.column_stack([rf_oof, lr_oof])

meta_model = xgb.XGBClassifier(random_state=42)
meta_model.fit(X_meta_train, y_train)


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [20]:
rf_best.fit(X_train_20, y_train)
lr_best.fit(X_train_20, y_train)

rf_test = rf_best.predict_proba(X_test_20)[:, 1]
lr_test = lr_best.predict_proba(X_test_20)[:, 1]

X_meta_test = np.column_stack([rf_test, lr_test])
stack_preds = meta_model.predict(X_meta_test)

stack_score = balanced_accuracy_score(y_test, stack_preds)
print("Stacked model balanced accuracy:", stack_score)


Stacked model balanced accuracy: 0.7435141539977891


In [21]:
final_summary = pd.DataFrame({
    "model": [
        "Tuned Random Forest",
        "Tuned Logistic Regression",
        "Stacked Model"
    ],
    "test_bal_acc": [
        balanced_accuracy_score(y_test, rf_best.predict(X_test_20)),
        balanced_accuracy_score(y_test, lr_best.predict(X_test_20)),
        stack_score
    ]
})

final_summary


,model,test_bal_acc
0,Tuned Random Forest,0.806603
1,Tuned Logistic Regression,0.788542
2,Stacked Model,0.743514


## Evaluation of results

I have the flu, so I didn't really do much exploration for this activity; the AI did most of the work.  Somehow, the meta-learner model actually got worse that the individual models, which is quite interesting (although I didn't tune the meta-learner).  As we saw during class, `marital status` was the most important feature by a long ways.  I arbitrarily removed the features whose average rank was outside of the top 20 (for random forest feature importance and permutation feature importance), which is likely why my models with 20 features performed poorly; marital status was in so many of the features.